# Chunk 35 — Functioning-classification metrics

## Pre-registered definitions
- **Label (`LABEL_MIN`):** y = 1 if the TRUE curve's minimum over 201 bins is < −10 dB. It is asserted equal to the stored `is_functioning`.
- **Score:** s = −(minimum of the PREDICTED curve, in dB). AUROC uses s directly.
- **Decision:** ŷ = 1 if the predicted minimum is < −10 dB. The threshold is **fixed by physics** (it is the label definition) and is never tuned on test.
- **Headline metrics:**
  - AUROC and **balanced accuracy** on the full natural-mix set, with 1,000 stratified bootstrap resamples (rng 351) for 95% intervals.
  - **Macro-F1** and **accuracy** on 1,000 **balanced subsamples**: the majority class is undersampled to the minority count, without replacement (rng 35).
- **Reported, never headlined:** raw (natural-mix) accuracy, prevalence, TPR, TNR, precision, natural-mix F1.
- **Secondary:** a Youden threshold fitted on VAL only and applied unchanged to TEST.

## Models
| model | notes |
|---|---|
| `best_model` | trained on 25×25 only; 35×35/45×45 are zero-shot |
| `backbone_no55` | Chunk 30 v2 replay backbone, 6 input features |
| `cnn_best` | Chunk 3 baseline. Its 25×25 headline uses **only test graphs outside the CNN's own training/val split**, because the CNN was trained on a different 25×25 split. |

## Scope and rules
- **Scopes:** grid 25 / 35 / 45, `POOL_FINETUNE` (35+45) and `POOL_EQUAL_GRID` (mean of the three grids). VAL and TEST are never mixed. Grid 55 is excluded everywhere.
- **No training, no optimizer, no test-based selection.** TEST is touched in exactly one cell, and only after the VAL harness reproduces Chunk 28.

**Changes from the submitted draft:**
- The draft had stub cells (staging, model loading, inference, metric loops, Youden, report) and a caption string with a syntax error.
- It hashed every `.pt` checkpoint (many GB); now only the files actually read are hashed, and every earlier artifact's mtime is captured.
- The CNN 25×25 contamination check is new.

**v2 additions:** a PAIRED comparison on identical graphs, and a report addendum with SECONDARY threshold calibration. **To update an existing run:** execute only the two added cells and the final guards. They read `ch35_per_sample.parquet`, and the earlier outputs are kept. In a fresh session, run the header, Setup and Splits cells first.

## Cell 1 — Installs

In [1]:
!pip -q install torch_geometric pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.1 MB/s eta 0:00:00


## Cell 2 — Repository and Drive
Clones or pulls the repo, puts `src/` on the path, and mounts Drive. It writes nothing.

In [2]:
import os, sys, subprocess
REPO_ROOT = '/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'):
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_ROOT], check=True)
sys.path.insert(0, f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
ART, CKPT, LOCAL = f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints', '/content/local_graphs'
os.makedirs(LOCAL, exist_ok=True)
print('repo:', REPO_ROOT, '| data:', DATA_ROOT)

Mounted at /content/drive
repo: /content/antenna_gnn | data: /content/drive/MyDrive/antenna_gnn


## Setup — immutable-state capture
**What it does:**
- SHA-256-hashes the files this notebook reads: the three checkpoints, the s11 statistics and the Chunk 28 cache.
- Records the mtimes of every earlier artifact and checkpoint.
- Defines the fixed threshold and the `ch35_`-only output helper.
- Loads the s11 statistics read-only and disables TF32.

**Writes:** nothing.

In [3]:
# ── Setup: immutable-state capture, constants ──
import json, glob, hashlib, time, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm
from model import AntennaGNN
from feed_component import AppendFeedComponentFeature

def digest(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

GRIDS = (25, 35, 45)
FIXED_TAU = -10.0          # physical threshold = the label definition; never tuned
TEST_TOUCHES = 0
CKPTS = {'best_model': f'{CKPT}/best_model.pt', 'backbone_no55': f'{CKPT}/backbone_no55.pt', 'cnn_best': f'{CKPT}/cnn_best.pt'}
# SHA-256 of the files this notebook READS (checkpoints, stats, Chunk 28 cache); mtimes of every earlier artifact.
# (Hashing every .pt under checkpoints/ would read many GB; only the inputs need a content proof.)
READ_FILES = list(CKPTS.values()) + [f'{ART}/s11_mean.npy', f'{ART}/s11_std.npy', f'{ART}/ch28_per_sample.parquet']
IMM_SHA = {p: digest(p) for p in READ_FILES}
IMM_MTIME = {p: os.path.getmtime(p) for p in glob.glob(f'{ART}/ch2[89]_*') + glob.glob(f'{ART}/ch30_*')
             + glob.glob(f'{ART}/ch31_*') + glob.glob(f'{CKPT}/*.pt') if os.path.isfile(p)}
print(f'hashed {len(IMM_SHA)} input files; captured mtimes of {len(IMM_MTIME)} earlier artifacts')

WRITTEN = []
def OUT(name):
    assert name.startswith('ch35_'), name
    p = f'{ART}/{name}'; WRITTEN.append(p); return p

torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); assert DEVICE.type == 'cuda', 'GPU required'
MEAN = torch.tensor(np.load(f'{ART}/s11_mean.npy'), dtype=torch.float32, device=DEVICE)   # read only
STD = torch.tensor(np.load(f'{ART}/s11_std.npy'), dtype=torch.float32, device=DEVICE)     # read only

hashed 6 input files; captured mtimes of 66 earlier artifacts


## Splits and the CNN contamination check
**What it does:**
- Builds VAL and TEST from `indices.json` (25×25) and the fine-tune val/test files (35/45), with grid 55 removed.
- Stops unless they equal Chunk 28's evaluated sets exactly.
- **CNN check:** `cnn_best` was trained on Chunk 3's own 25×25 split. Any `indices.json` test graph that is in that split's train or val portion is **in-sample** for the CNN. The cell counts them and defines the clean subset used for the CNN's 25×25 headline.
  - **Stated assumption:** Chunk 3's dataset index equals `local_idx`.

**Writes:** nothing.

In [4]:
# ── Splits (grid 55 removed) reconciled against the Chunk 28 cache ──
P28 = pd.read_parquet(f'{ART}/ch28_per_sample.parquet')
P28 = P28[(P28.model_id == 'best_model') & P28.grid_size.isin(GRIDS)].copy()
assert not (P28.grid_size == 55).any()
def pairs(x):
    if isinstance(x, dict):
        return {(int(g), int(i)) for g, v in x.items() for i in v}
    return {(int(e[0]), int(e[1])) for e in x}
idx = json.load(open(f'{DATA_ROOT}/splits/indices.json'))
ft = {r: json.load(open(f'{DATA_ROOT}/splits/finetune_{r}_indices.json')) for r in ('val', 'test')}
S = {'VAL': {p for p in pairs(idx['val']) if p[0] == 25} | {p for p in pairs(ft['val']) if p[0] in (35, 45)},
     'TEST': {p for p in pairs(idx['test']) if p[0] == 25} | {p for p in pairs(ft['test']) if p[0] in (35, 45)}}
assert not (S['VAL'] & S['TEST'])
for role, sp in {'VAL': ('25x25_val', 'finetune_val'), 'TEST': ('25x25_test', 'finetune_test')}.items():
    got = {(int(g), int(i)) for g, i in P28[P28.split.isin(sp)][['grid_size', 'local_idx']].values}
    assert got == S[role], f'STOP: Chunk 28 split mismatch for {role}: {len(got ^ S[role])} keys differ'
print({r: {g: sum(p[0] == g for p in v) for g in GRIDS} for r, v in S.items()})

# CNN contamination check. cnn_best was trained on its OWN 25x25 split (Chunk 3, cnn_split_indices.json,
# indices into the 25x25 dataset). Any indices.json test graph that is in the CNN's train/val split is
# IN-SAMPLE for the CNN. ASSUMPTION (stated in the report): Chunk 3's dataset index equals local_idx.
cnn_split = json.load(open(f'{DATA_ROOT}/splits/cnn_split_indices.json'))
cnn_seen = set(map(int, cnn_split['train'])) | set(map(int, cnn_split['val']))
t25 = {i for g, i in S['TEST'] if g == 25}; v25 = {i for g, i in S['VAL'] if g == 25}
CNN_CLEAN = {'TEST': {(25, i) for i in t25 - cnn_seen}, 'VAL': {(25, i) for i in v25 - cnn_seen}}
print(f'25x25 TEST: {len(t25 & cnn_seen):,} of {len(t25):,} graphs were in the CNN\'s training/val split; '
      f'clean CNN subset = {len(CNN_CLEAN["TEST"]):,}')
print(f'25x25 VAL : {len(v25 & cnn_seen):,} of {len(v25):,} in the CNN\'s training/val split; clean = {len(CNN_CLEAN["VAL"]):,}')

{'VAL': {25: 9984, 35: 499, 45: 698}, 'TEST': {25: 9983, 35: 499, 45: 699}}
25x25 TEST: 8,935 of 9,983 graphs were in the CNN's training/val split; clean CNN subset = 1,048
25x25 VAL : 8,975 of 9,984 in the CNN's training/val split; clean = 1,009


## Stage graphs
Unzips `data/processed/processed_{N}x{N}.zip` flat to local SSD, as in Chunk 30, then checks on the files themselves that `y` is raw dB.

In [5]:
# ── Stage graphs to local SSD (flat sample_<i>.pt), as in Chunk 30 ──
def stage(g):
    d = f'{LOCAL}/{g}x{g}'
    if os.path.exists(f'{d}/_STAGED.txt'):
        return d
    z = f'{DATA_ROOT}/data/processed/processed_{g}x{g}.zip'
    assert os.path.exists(z), f'missing {z}'
    os.makedirs(d, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        mem = [m for m in zf.namelist() if m.endswith('.pt')]
        for m in tqdm(mem, desc=f'unzip {g}x{g}', leave=False):
            with zf.open(m) as src, open(f'{d}/{os.path.basename(m)}', 'wb') as dst:
                shutil.copyfileobj(src, dst)
    open(f'{d}/_STAGED.txt', 'w').write(f'{len(mem)}\n')
    return d
for g in GRIDS:
    stage(g)
def load_graph(g, i):
    return torch.load(f'{LOCAL}/{g}x{g}/sample_{i}.pt', weights_only=False)
for g in GRIDS:   # identity guard on the on-disk files: y is raw dB
    for p in sorted(glob.glob(f'{LOCAL}/{g}x{g}/sample_*.pt'))[:5]:
        y = torch.load(p, weights_only=False).y
        assert y.shape[-1] == 201 and float(y.max()) <= 0.01, f'{p}: y is not raw dB'
print('staged; on-disk y is raw dB')

unzip 25x25:   0%|          | 0/99833 [00:00<?, ?it/s]

unzip 35x35:   0%|          | 0/4988 [00:00<?, ?it/s]

unzip 45x45:   0%|          | 0/6984 [00:00<?, ?it/s]

staged; on-disk y is raw dB


## Models and inference
**What it does:**
- Copies the CNN verbatim from Chunk 3. Its output is raw dB.
- Loads the three checkpoints strictly, stripping any `module.` prefix, and asserts both the trainable-parameter count and the state_dict total: 587,001 and 587,129 for the GNNs (no buffers); 6,163,473 parameters and 6,172,251 state values for the CNN (BatchNorm buffers account for the difference).
- `infer()` does one pass over the graphs, with all three models predicting each batch:
  - GNN outputs are de-normalised with the read-only statistics.
  - `backbone_no55` receives the feed-component feature.
  - The CNN receives the N×N metal pattern.
  - Batches are single-grid; eval mode, `no_grad`, batch 64.

In [6]:
# ── The three frozen models ──
# frozen — validated in Chunk 3 (AntennaCNN, copied verbatim; output is RAW dB, no de-normalisation)
class AntennaCNN(nn.Module):
    """
    Adapted from Gupta et al. IEEE TAP 2023, Fig. 4.
    Original: 12x12 input, 81 output points.
    Adapted: NxN input via AdaptiveAvgPool2d, 201 output points (1-4 GHz).
    """
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )

        layers2 = []
        layers2.extend([nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        for _ in range(4):
            layers2.extend([nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        self.block2 = nn.Sequential(*layers2)

        layers3 = []
        layers3.extend([nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        for _ in range(7):
            layers3.extend([nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        self.block3 = nn.Sequential(*layers3)

        self.readout = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 1000),
            nn.BatchNorm1d(1000),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(1000, 500),
            nn.BatchNorm1d(500),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(500, 201)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.readout(x)
        x = self.fc(x)
        return x

def state_of(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    st = ck['model_state'] if isinstance(ck, dict) and 'model_state' in ck else ck
    return {k[7:] if k.startswith('module.') else k: v for k, v in st.items()}   # strip DataParallel prefix

def gnn(nf):
    return AntennaGNN(node_feat_dim=nf, edge_feat_dim=2, hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201)
MODELS = {}
m = gnn(5); m.load_state_dict(state_of(CKPTS['best_model']), strict=True); MODELS['best_model'] = m
m = gnn(6); m.load_state_dict(state_of(CKPTS['backbone_no55']), strict=True); MODELS['backbone_no55'] = m
m = AntennaCNN(); m.load_state_dict(state_of(CKPTS['cnn_best']), strict=True); MODELS['cnn_best'] = m
# Trainable parameters, and the full state_dict (parameters + buffers). The CNN's 18 BatchNorm layers add
# 8,778 buffer values (running mean/var for 4,380 channels + 18 num_batches_tracked): 6,163,473 trainable
# parameters, 6,172,251 state_dict values. The earlier 6,172,251 "parameter" figure was the state_dict total.
# The GNNs have no buffers, so both counts agree. strict=True loading above already proves the architecture.
EXPECT = {'best_model': (587_001, 587_001), 'backbone_no55': (587_129, 587_129), 'cnn_best': (6_163_473, 6_172_251)}
for k, m in MODELS.items():
    n_par = sum(p.numel() for p in m.parameters())
    n_sd = sum(v.numel() for v in m.state_dict().values())
    assert (n_par, n_sd) == EXPECT[k], f'{k}: {n_par:,} params / {n_sd:,} state values, expected {EXPECT[k]}'
    MODELS[k] = m.to(DEVICE).eval()
print('models loaded:', {k: f'{EXPECT[k][0]:,} params ({EXPECT[k][1]:,} state values)' for k in MODELS})
TRANSFORM = AppendFeedComponentFeature()

@torch.no_grad()
def infer(role, keys, bs=64):
    """One pass over the graphs; all three models predict each batch. Returns per-sample rows.
    Batches are single-grid (the CNN stacks N x N images)."""
    rows = []
    for g in tqdm(GRIDS, desc=f'{role} progress by grid'):
        kg = sorted(k for k in keys if k[0] == g)
        for s in tqdm(range(0, len(kg), bs), desc=f'{role} batches {g}x{g}', leave=False):
            chunk = kg[s:s + bs]
            ds = [load_graph(g, i) for _, i in chunk]
            ytrue = np.stack([d.y.reshape(-1).numpy() for d in ds]).astype(np.float64)
            assert ytrue.shape[1] == 201 and ytrue.max() <= 0.01
            plain = [Data(x=d.x, edge_index=d.edge_index, edge_attr=d.edge_attr) for d in ds]
            b5 = next(iter(DataLoader(plain, batch_size=len(plain)))).to(DEVICE)
            b6 = next(iter(DataLoader([TRANSFORM(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr))
                                       for d in ds], batch_size=len(ds)))).to(DEVICE)
            img = torch.stack([d.x[:-1, 0].reshape(1, g, g) for d in ds]).float().to(DEVICE)
            preds = {'best_model': (MODELS['best_model'](b5) * STD + MEAN),
                     'backbone_no55': (MODELS['backbone_no55'](b6) * STD + MEAN),
                     'cnn_best': MODELS['cnn_best'](img)}
            for mid, P in preds.items():
                pm = P.min(1).values.double().cpu().numpy()
                for k, (gg, i) in enumerate(chunk):
                    tm = float(ytrue[k].min())
                    rows.append(dict(model_id=mid, role=role, grid=gg, local_idx=int(i), true_min_db=tm,
                                     pred_min_db=float(pm[k]), y=int(tm < FIXED_TAU), y_hat=int(pm[k] < FIXED_TAU)))
    return rows

models loaded: {'best_model': '587,001 params (587,001 state values)', 'backbone_no55': '587,129 params (587,129 state values)', 'cnn_best': '6,163,473 params (6,172,251 state values)'}


## VAL predictions and the harness
**What it does:**
- Predicts all VAL graphs.
- **Harness:** `best_model`'s predicted minima must match the Chunk 28 cache to < 1e-3 dB for ≥ 99.9% of graphs, which proves the pipeline reproduces Chunk 28.
- Asserts `LABEL_MIN` equals the stored `is_functioning`.

The TEST cell refuses to run unless this passes.

In [7]:
# ── VAL predictions and the harness check (before any TEST access) ──
VAL_ROWS = infer('VAL', S['VAL'])
V = pd.DataFrame(VAL_ROWS)
ref = P28[P28.split.isin(['25x25_val', 'finetune_val'])].set_index(['grid_size', 'local_idx'])
bm = V[V.model_id == 'best_model'].set_index(['grid', 'local_idx'])
gap = (bm['pred_min_db'] - ref.loc[bm.index, 'pred_min_db'].values).abs()
share = float((gap < 1e-3).mean())
print(f'HARNESS: best_model VAL pred_min within 1e-3 dB of Chunk 28 for {share*100:.3f}% '
      f'of {len(gap):,} graphs (max |diff| {gap.max():.2e} dB)')
assert share >= 0.999, 'STOP: harness does not reproduce Chunk 28; do not run the TEST TOUCH cell'
# labels must equal the stored is_functioning (Chunk 28 cache)
lab = ref.loc[bm.index, 'is_functioning'].astype(int).values
assert (bm['y'].values == lab).all(), 'STOP: LABEL_MIN differs from the stored is_functioning'
HARNESS_OK = True
print('VAL done; harness PASSED; labels match the stored is_functioning.')

VAL progress by grid:   0%|          | 0/3 [00:00<?, ?it/s]

VAL batches 25x25:   0%|          | 0/156 [00:00<?, ?it/s]

VAL batches 35x35:   0%|          | 0/8 [00:00<?, ?it/s]

VAL batches 45x45:   0%|          | 0/11 [00:00<?, ?it/s]

HARNESS: best_model VAL pred_min within 1e-3 dB of Chunk 28 for 100.000% of 11,181 graphs (max |diff| 7.44e-05 dB)
VAL done; harness PASSED; labels match the stored is_functioning.


## TEST TOUCH
This is the **only** cell that loads or scores test graphs, guarded by a counter.

**What it does:**
- Runs the same frozen inference.
- Asserts the test labels equal the stored `is_functioning`.
- Flags the CNN's clean rows.

**Writes:** `ch35_per_sample.parquet`.

In [8]:
# ── TEST TOUCH: the only cell that loads or scores test graphs ──
assert globals().get('HARNESS_OK') is True, 'STOP: run the VAL harness first'
TEST_TOUCHES += 1
assert TEST_TOUCHES == 1, 'TEST may be produced only once, in this cell'
TEST_ROWS = infer('TEST', S['TEST'])
ALL = pd.DataFrame(VAL_ROWS + TEST_ROWS)
assert not (ALL.grid == 55).any()
tref = P28[P28.split.isin(['25x25_test', 'finetune_test'])].set_index(['grid_size', 'local_idx'])
tb = ALL[(ALL.role == 'TEST') & (ALL.model_id == 'best_model')].set_index(['grid', 'local_idx'])
assert (tb['y'].values == tref.loc[tb.index, 'is_functioning'].astype(int).values).all()
ALL['cnn_clean'] = [(r.model_id != 'cnn_best') or (r.grid != 25) or ((r.grid, r.local_idx) in CNN_CLEAN[r.role])
                    for r in ALL.itertuples()]
ALL.to_parquet(OUT('ch35_per_sample.parquet'), index=False)
print(f'TEST touched once; {len(TEST_ROWS):,} rows. Wrote ch35_per_sample.parquet. Threshold fixed at {FIXED_TAU} dB.')

TEST progress by grid:   0%|          | 0/3 [00:00<?, ?it/s]

TEST batches 25x25:   0%|          | 0/156 [00:00<?, ?it/s]

TEST batches 35x35:   0%|          | 0/8 [00:00<?, ?it/s]

TEST batches 45x45:   0%|          | 0/11 [00:00<?, ?it/s]

TEST touched once; 33,543 rows. Wrote ch35_per_sample.parquet. Threshold fixed at -10.0 dB.


## Metrics
**What it does:** for each model × role × scope, computes every pre-registered metric with its interval.
- **`POOL_EQUAL_GRID`:** each grid is resampled separately, and the replicates are averaged across grids.
- **`cnn_best`:** reported twice. `clean` (outside its own training split) is the headline; `all_IN_SAMPLE_25` is shown only for transparency.

VAL and TEST are never mixed.

In [9]:
# ── Pre-registered metrics with intervals (never mixing VAL and TEST) ──
R = 1000
def counts(y, yh):
    tp = int(((y == 1) & (yh == 1)).sum()); tn = int(((y == 0) & (yh == 0)).sum())
    fp = int(((y == 0) & (yh == 1)).sum()); fn = int(((y == 1) & (yh == 0)).sum())
    return tp, tn, fp, fn
def bal_acc(y, yh):
    tp, tn, fp, fn = counts(y, yh); return 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
def macro_f1(y, yh):
    tp, tn, fp, fn = counts(y, yh)
    f1p = 2 * tp / max(2 * tp + fp + fn, 1); f1n = 2 * tn / max(2 * tn + fn + fp, 1)
    return 0.5 * (f1p + f1n)

def point(d):
    y, yh, s = d.y.values, d.y_hat.values, -d.pred_min_db.values
    tp, tn, fp, fn = counts(y, yh)
    return dict(auroc=roc_auc_score(y, s), balanced_accuracy=bal_acc(y, yh), raw_accuracy=float((y == yh).mean()),
                prevalence=float(y.mean()), TPR=tp / max(tp + fn, 1), TNR=tn / max(tn + fp, 1),
                precision=tp / max(tp + fp, 1), f1_natural=2 * tp / max(2 * tp + fp + fn, 1),
                n=len(y), n_pos=int(y.sum()), n_neg=int((1 - y).sum()))

def boot_reps(d, rng):
    """R stratified bootstrap replicates of (AUROC, balanced accuracy) for one group."""
    y, yh, s = d.y.values, d.y_hat.values, -d.pred_min_db.values
    i0, i1 = np.flatnonzero(y == 0), np.flatnonzero(y == 1); out = np.empty((R, 2))
    for r in tqdm(range(R), desc='Stratified bootstrap', leave=False):
        ix = np.r_[rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]
        out[r] = (roc_auc_score(y[ix], s[ix]), bal_acc(y[ix], yh[ix]))
    return out

def balanced_reps(d, rng):
    """R balanced subsamples (majority undersampled to the minority count, no replacement):
    (macro-F1, accuracy)."""
    y, yh = d.y.values, d.y_hat.values
    i0, i1 = np.flatnonzero(y == 0), np.flatnonzero(y == 1); n = min(len(i0), len(i1)); out = np.empty((R, 2))
    for r in tqdm(range(R), desc='Balanced subsamples', leave=False):
        ix = np.r_[rng.choice(i0, n, replace=False), rng.choice(i1, n, replace=False)]
        out[r] = (macro_f1(y[ix], yh[ix]), float((y[ix] == yh[ix]).mean()))
    return out

def scopes(d):
    return {'grid_25': [d[d.grid == 25]], 'grid_35': [d[d.grid == 35]], 'grid_45': [d[d.grid == 45]],
            'POOL_FINETUNE': [d[d.grid.isin([35, 45])]],
            'POOL_EQUAL_GRID': [d[d.grid == g] for g in GRIDS]}

ROWS = []
for (mid, role), d in tqdm(ALL.groupby(['model_id', 'role']), desc='Computing metrics'):
    variants = [('all', d)]
    if mid == 'cnn_best':
        variants = [('all_IN_SAMPLE_25', d), ('clean', d[d.cnn_clean])]   # CNN 25x25 headline = clean subset
    for subset, dd in variants:
        for scope, parts in scopes(dd).items():
            parts = [p for p in parts if len(p)]
            if not parts:
                continue
            rb, rs = np.random.default_rng(351), np.random.default_rng(35)
            pts = [point(p) for p in parts]
            br = np.mean([boot_reps(p, rb) for p in parts], axis=0)        # equal-grid: average replicates
            sr = np.mean([balanced_reps(p, rs) for p in parts], axis=0)
            est = {k: float(np.mean([q[k] for q in pts])) for k in pts[0] if k not in ('n', 'n_pos', 'n_neg')}
            est.update(macro_f1_balanced=float(sr[:, 0].mean()), accuracy_balanced=float(sr[:, 1].mean()))
            iv = {'auroc': br[:, 0], 'balanced_accuracy': br[:, 1], 'macro_f1_balanced': sr[:, 0], 'accuracy_balanced': sr[:, 1]}
            for k, v in est.items():
                lo, hi = (np.quantile(iv[k], [0.025, 0.975]) if k in iv else (np.nan, np.nan))
                ROWS.append(dict(model_id=mid, subset=subset, role=role, scope=scope, metric=k, estimate=v, lo=lo, hi=hi,
                                 n=sum(q['n'] for q in pts), n_pos=sum(q['n_pos'] for q in pts),
                                 n_neg=sum(q['n_neg'] for q in pts), threshold='fixed_-10dB'))
MET = pd.DataFrame(ROWS)
print(MET[(MET.metric.isin(['auroc', 'balanced_accuracy', 'macro_f1_balanced'])) & (MET.role == 'TEST')]
      .pivot_table(index=['model_id', 'subset', 'scope'], columns='metric', values='estimate').round(4).to_string())

Computing metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Stratified bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

Balanced subsamples:   0%|          | 0/1000 [00:00<?, ?it/s]

Stratified bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

Balanced subsamples:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Stratified bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

Balanced subsamples:   0%|          | 0/1000 [00:00<?, ?it/s]

Balanced subsamples:   0%|          | 0/1000 [00:00<?, ?it/s]

Balanced subsamples:   0%|          | 0/1000 [00:00<?, ?it/s]

metric                                           auroc  balanced_accuracy  macro_f1_balanced
model_id      subset           scope                                                        
backbone_no55 all              POOL_EQUAL_GRID  0.9327             0.8609             0.8587
                               POOL_FINETUNE    0.9466             0.8804             0.8794
                               grid_25          0.9052             0.8254             0.8215
                               grid_35          0.9427             0.8671             0.8656
                               grid_45          0.9502             0.8901             0.8889
best_model    all              POOL_EQUAL_GRID  0.9240             0.8272             0.8227
                               POOL_FINETUNE    0.9347             0.8254             0.8206
                               grid_25          0.9040             0.8259             0.8218
                               grid_35          0.9289             0.8

## SECONDARY — VAL-fitted Youden threshold
**What it does:** for each model and scope, picks τ on VAL by maximising Youden's J (TPR − FPR), then applies it **unchanged** to TEST. These results are labelled SECONDARY and never replace the −10 dB headline.

**Writes:** `ch35_metrics.csv`, containing the headline and secondary rows.

In [10]:
# ── SECONDARY: Youden threshold fitted on VAL only, applied unchanged to TEST ──
SEC = []
for mid in ALL.model_id.unique():
    for scope in ('grid_25', 'grid_35', 'grid_45', 'POOL_FINETUNE'):
        grids = {'grid_25': [25], 'grid_35': [35], 'grid_45': [45], 'POOL_FINETUNE': [35, 45]}[scope]
        sel = (ALL.model_id == mid) & ALL.grid.isin(grids) & ALL.cnn_clean
        v, t = ALL[sel & (ALL.role == 'VAL')], ALL[sel & (ALL.role == 'TEST')]
        if len(v) == 0 or len(t) == 0:
            continue
        fpr, tpr, thr = roc_curve(v.y, -v.pred_min_db)          # score = -pred_min
        tau = -float(thr[np.argmax(tpr - fpr)])                 # back to a dB threshold on pred_min
        yh = (t.pred_min_db.values < tau).astype(int)
        for k, val in (('balanced_accuracy', bal_acc(t.y.values, yh)), ('macro_f1_natural', macro_f1(t.y.values, yh))):
            SEC.append(dict(model_id=mid, subset='clean' if mid == 'cnn_best' else 'all', role='TEST', scope=scope,
                            metric=k, estimate=val, lo=np.nan, hi=np.nan, n=len(t), n_pos=int(t.y.sum()),
                            n_neg=int((1 - t.y).sum()), threshold=f'SECONDARY_youden_val={tau:.3f}dB'))
assert (ALL.y_hat == (ALL.pred_min_db < FIXED_TAU)).all(), 'headline decisions must use -10 dB'
MET_ALL = pd.concat([MET, pd.DataFrame(SEC)], ignore_index=True)
MET_ALL.to_csv(OUT('ch35_metrics.csv'), index=False)
print(pd.DataFrame(SEC).round(4).to_string(index=False))

     model_id subset role         scope            metric  estimate  lo  hi    n  n_pos  n_neg                     threshold
   best_model    all TEST       grid_25 balanced_accuracy    0.8286 NaN NaN 9983   5695   4288 SECONDARY_youden_val=-7.699dB
   best_model    all TEST       grid_25  macro_f1_natural    0.8131 NaN NaN 9983   5695   4288 SECONDARY_youden_val=-7.699dB
   best_model    all TEST       grid_35 balanced_accuracy    0.8658 NaN NaN  499    318    181 SECONDARY_youden_val=-2.842dB
   best_model    all TEST       grid_35  macro_f1_natural    0.8513 NaN NaN  499    318    181 SECONDARY_youden_val=-2.842dB
   best_model    all TEST       grid_45 balanced_accuracy    0.8961 NaN NaN  699    458    241 SECONDARY_youden_val=-3.175dB
   best_model    all TEST       grid_45  macro_f1_natural    0.8684 NaN NaN  699    458    241 SECONDARY_youden_val=-3.175dB
   best_model    all TEST POOL_FINETUNE balanced_accuracy    0.8828 NaN NaN 1198    776    422 SECONDARY_youden_val=-3.175dB


## Comparison report
**What it does:** writes the TEST headline table with 95% intervals, per grid and `POOL_EQUAL_GRID`.
- The Rather et al. row is labelled **REPORTED BY THE AUTHORS**: arXiv:2608.23469, Table I.
- The coherence notes go in the caption.
- For each model it states whether its AUROC interval lies entirely above 0.874, and its balanced-subsample macro-F1 interval entirely above 0.796.

**Writes:** `ch35_comparison.md`.

In [11]:
# ── Comparison report (TEST, fixed -10 dB threshold) ──
RATHER = {'Random Forest': (0.801, 0.796, 0.874), 'XGBoost': (0.774, 0.768, 0.844)}
def get(mid, subset, scope, metric):
    r = MET[(MET.model_id == mid) & (MET.subset == subset) & (MET.role == 'TEST') & (MET.scope == scope) & (MET.metric == metric)]
    return r.iloc[0] if len(r) else None
def fmt(r):
    return '—' if r is None else f'{r.estimate:.3f} [{r.lo:.3f}, {r.hi:.3f}]'
L = ['# Chunk 35 — Functioning classification (TEST, fixed −10 dB threshold)', '',
     '| model | scope | n (pos/neg) | AUROC | balanced acc. | macro-F1 (balanced) | acc. (balanced) |',
     '|---|---|---|---|---|---|---|']
for mid, subset in (('best_model', 'all'), ('backbone_no55', 'all'), ('cnn_best', 'clean')):
    for scope in ('grid_25', 'grid_35', 'grid_45', 'POOL_EQUAL_GRID'):
        a = get(mid, subset, scope, 'auroc')
        if a is None:
            continue
        L.append(f'| {mid} | {scope} | {a.n} ({a.n_pos}/{a.n_neg}) | {fmt(a)} | {fmt(get(mid, subset, scope, "balanced_accuracy"))} | '
                 f'{fmt(get(mid, subset, scope, "macro_f1_balanced"))} | {fmt(get(mid, subset, scope, "accuracy_balanced"))} |')
for name, (acc, f1, auc) in RATHER.items():
    L.append(f'| Rather et al. 2026, {name} — REPORTED BY THE AUTHORS | balanced test set | — | {auc:.3f} | — | {f1:.3f} | {acc:.3f} |')
L += ['',
      '*Coherence notes.* Same label definition (min |S11| < −10 dB anywhere in the band). Same input type (binary pixel grid). '
      'Their test set is balanced, so it is compared with OUR balanced-subsample accuracy and macro-F1, with balanced accuracy, '
      'and with AUROC; never with our natural-mix accuracy. Differences: their 19×23 grid on RT/duroid 5880 at 22–30 GHz, '
      'feed-connected structured-random patterns, ~4k balanced samples, and tabular classifiers (RF/XGBoost) used as a '
      'standalone screener; our decision is derived from a predicted spectrum, and our patterns are unconstrained random '
      'fills (isolated feeds included). best_model is zero-shot at 35×35/45×45. cnn_best 25×25 rows use only test graphs '
      'outside the CNN\'s own training/validation split (assumes Chunk 3 dataset index = local_idx); all other rows use the '
      'full test set. Rather et al. values are as reported in arXiv:2608.23469, Table I, not re-computed.', '']
L.append('## Interval comparison against Rather et al. (best: RF, AUC 0.874, macro-F1 0.796)')
for mid, subset in (('best_model', 'all'), ('backbone_no55', 'all'), ('cnn_best', 'clean')):
    for scope in ('grid_25', 'grid_35', 'grid_45', 'POOL_EQUAL_GRID'):
        a, f = get(mid, subset, scope, 'auroc'), get(mid, subset, scope, 'macro_f1_balanced')
        if a is None:
            continue
        L.append(f'- {mid} {scope}: AUROC interval entirely above 0.874: {"YES" if a.lo > 0.874 else "NO"}; '
                 f'balanced-subsample macro-F1 interval entirely above 0.796: {"YES" if f.lo > 0.796 else "NO"}')
L += ['', '_Secondary (VAL-fitted Youden threshold) results are in ch35_metrics.csv and never replace the headline._']
open(OUT('ch35_comparison.md'), 'w').write('\n'.join(L) + '\n')
print('\n'.join(L))

# Chunk 35 — Functioning classification (TEST, fixed −10 dB threshold)

| model | scope | n (pos/neg) | AUROC | balanced acc. | macro-F1 (balanced) | acc. (balanced) |
|---|---|---|---|---|---|---|
| best_model | grid_25 | 9983 (5695/4288) | 0.904 [0.898, 0.910] | 0.826 [0.819, 0.832] | 0.822 [0.818, 0.826] | 0.826 [0.823, 0.829] |
| best_model | grid_35 | 499 (318/181) | 0.929 [0.905, 0.951] | 0.840 [0.811, 0.868] | 0.837 [0.816, 0.860] | 0.840 [0.820, 0.862] |
| best_model | grid_45 | 699 (458/241) | 0.939 [0.922, 0.955] | 0.816 [0.794, 0.837] | 0.809 [0.786, 0.832] | 0.815 [0.795, 0.836] |
| best_model | POOL_EQUAL_GRID | 11181 (6471/4710) | 0.924 [0.913, 0.933] | 0.827 [0.814, 0.839] | 0.823 [0.811, 0.834] | 0.827 [0.817, 0.837] |
| backbone_no55 | grid_25 | 9983 (5695/4288) | 0.905 [0.900, 0.910] | 0.825 [0.819, 0.832] | 0.821 [0.818, 0.825] | 0.825 [0.822, 0.829] |
| backbone_no55 | grid_35 | 499 (318/181) | 0.943 [0.924, 0.961] | 0.867 [0.840, 0.893] | 0.866 [0.846, 0.886] | 0.8

## PAIRED comparison on identical graphs *(added)*
**Why:** in the headline table, the CNN's 25×25 row uses the 1,048 clean graphs while the GNN rows use all 9,983. GNN-vs-CNN at 25×25, and therefore in `POOL_EQUAL_GRID`, compared **different samples**.

**What it does:** scores all three models on exactly the same graphs: the 25×25 clean subset plus the full 35×35 and 45×45 test sets. For each pair of models it reports the difference with a **paired** bootstrap (the same resampled graphs for both models) and the same balanced subsamples. Each difference gets a verdict: A better, B better, or no significant difference.

It reads `ch35_per_sample.parquet`, so it runs **no new inference and is not a test touch**.

**Writes:** `ch35_paired.csv`.

In [12]:
# ── PAIRED comparison on identical graphs (fixes the 25x25 mismatch in the headline table) ──
# The headline table's cnn_best 25x25 row uses the 1,048 clean graphs while the GNN rows use all
# 9,983, so GNN-vs-CNN at 25x25 (and in POOL_EQUAL_GRID) compared different samples. Here every model
# is scored on EXACTLY the same graphs: 25x25 clean subset (outside the CNN's own training split),
# plus the full 35x35 and 45x45 test sets. Differences use a PAIRED bootstrap (the same resampled
# graphs for both models) and the same balanced subsamples. No new inference: reads
# ch35_per_sample.parquet, so this is not a test touch.
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D = pd.read_parquet(f'{ART}/ch35_per_sample.parquet')
D = D[D.role == 'TEST']
clean25 = set(map(tuple, D[(D.model_id == 'cnn_best') & (D.grid == 25) & D.cnn_clean][['grid', 'local_idx']].values))
keep = lambda d: d[(d.grid != 25).values | np.array([k in clean25 for k in zip(d.grid, d.local_idx)], dtype=bool)]
W = {m: keep(D[D.model_id == m]).set_index(['grid', 'local_idx']).sort_index() for m in ('best_model', 'backbone_no55', 'cnn_best')}
ref_index = W['cnn_best'].index
for m in W:
    assert W[m].index.equals(ref_index), f'{m}: graph set differs from cnn_best'
    assert (W[m]['y'].values == W['cnn_best']['y'].values).all()
print('paired graph set:', {g: int((ref_index.get_level_values(0) == g).sum()) for g in (25, 35, 45)})

R_P = 1000
def _ba(y, yh):
    tp = ((y == 1) & (yh == 1)).sum(); tn = ((y == 0) & (yh == 0)).sum()
    return 0.5 * (tp / max((y == 1).sum(), 1) + tn / max((y == 0).sum(), 1))
def _mf1(y, yh):
    tp = ((y == 1) & (yh == 1)).sum(); tn = ((y == 0) & (yh == 0)).sum()
    fp = ((y == 0) & (yh == 1)).sum(); fn = ((y == 1) & (yh == 0)).sum()
    return 0.5 * (2 * tp / max(2 * tp + fp + fn, 1) + 2 * tn / max(2 * tn + fn + fp, 1))

def reps_for_grid(g, models, rng_b, rng_s):
    """Per-replicate metric values for each model on grid g, with SHARED resampling across models."""
    y = W['cnn_best'].loc[g]['y'].values
    s = {m: -W[m].loc[g]['pred_min_db'].values for m in models}
    yh = {m: W[m].loc[g]['y_hat'].values for m in models}
    i0, i1 = np.flatnonzero(y == 0), np.flatnonzero(y == 1); n = min(len(i0), len(i1))
    out = {m: np.empty((R_P, 3)) for m in models}
    for r in range(R_P):
        ib = np.r_[rng_b.choice(i0, len(i0)), rng_b.choice(i1, len(i1))]          # bootstrap (AUROC, BA)
        isub = np.r_[rng_s.choice(i0, n, replace=False), rng_s.choice(i1, n, replace=False)]   # balanced subsample
        for m in models:
            out[m][r] = (roc_auc_score(y[ib], s[m][ib]), _ba(y[ib], yh[m][ib]), _mf1(y[isub], yh[m][isub]))
    point = {m: (roc_auc_score(y, s[m]), _ba(y, yh[m]), float(np.mean(out[m][:, 2]))) for m in models}
    return out, point

MODELS_P = ('best_model', 'backbone_no55', 'cnn_best')
NAMES = ('auroc', 'balanced_accuracy', 'macro_f1_balanced')
rb, rs = np.random.default_rng(3501), np.random.default_rng(3502)
REPS, PTS = {}, {}
for g in (25, 35, 45):
    REPS[g], PTS[g] = reps_for_grid(g, MODELS_P, rb, rs)
scopes_p = {'grid_25_clean': [25], 'grid_35': [35], 'grid_45': [45], 'POOL_EQUAL_GRID_paired': [25, 35, 45]}
rows = []
for scope, gs in scopes_p.items():
    for m in MODELS_P:
        rep = np.mean([REPS[g][m] for g in gs], axis=0); pt = np.mean([PTS[g][m] for g in gs], axis=0)
        for j, k in enumerate(NAMES):
            lo, hi = np.quantile(rep[:, j], [0.025, 0.975])
            rows.append(dict(kind='value', scope=scope, model=m, metric=k, estimate=pt[j], lo=lo, hi=hi))
    for a, b in (('backbone_no55', 'cnn_best'), ('best_model', 'cnn_best'), ('backbone_no55', 'best_model')):
        rep = np.mean([REPS[g][a] - REPS[g][b] for g in gs], axis=0)
        pt = np.mean([np.subtract(PTS[g][a], PTS[g][b]) for g in gs], axis=0)
        for j, k in enumerate(NAMES):
            lo, hi = np.quantile(rep[:, j], [0.025, 0.975])
            rows.append(dict(kind='difference', scope=scope, model=f'{a} - {b}', metric=k, estimate=pt[j], lo=lo, hi=hi,
                             verdict='A better' if lo > 0 else ('B better' if hi < 0 else 'no significant difference')))
PAIRED = pd.DataFrame(rows)
PAIRED.to_csv(f'{ART}/ch35_paired.csv', index=False)
if 'WRITTEN' in globals(): WRITTEN.append(f'{ART}/ch35_paired.csv')
pd.set_option('display.width', 220)
print(PAIRED[PAIRED.kind == 'difference'].round(4).to_string(index=False))

paired graph set: {25: 1048, 35: 499, 45: 699}
      kind                  scope                      model            metric  estimate      lo      hi                   verdict
difference          grid_25_clean   backbone_no55 - cnn_best             auroc    0.0030 -0.0036  0.0102 no significant difference
difference          grid_25_clean   backbone_no55 - cnn_best balanced_accuracy   -0.0179 -0.0282 -0.0068                  B better
difference          grid_25_clean   backbone_no55 - cnn_best macro_f1_balanced   -0.0180 -0.0229 -0.0132                  B better
difference          grid_25_clean      best_model - cnn_best             auroc   -0.0012 -0.0084  0.0063 no significant difference
difference          grid_25_clean      best_model - cnn_best balanced_accuracy   -0.0095 -0.0197  0.0012 no significant difference
difference          grid_25_clean      best_model - cnn_best macro_f1_balanced   -0.0098 -0.0141 -0.0055                  B better
difference          grid_25_clean ba

## Report addendum *(added)*
**What it does:** appends two sections to `ch35_comparison.md`, replacing any earlier copies of them:
1. **The paired comparison table.** This, not the headline table, is the basis for any GNN-vs-CNN claim.
2. **SECONDARY threshold calibration.** VAL-fitted Youden thresholds fall around −3 to −8 dB, not −10, showing that predicted minima are too shallow. The section sets balanced accuracy at −10 dB against balanced accuracy at the VAL-fitted threshold.

It also adds a note that the 89.5% overlap does not by itself validate the CNN index-mapping assumption.

In [13]:
# ── Report addendum: paired comparison + threshold calibration (secondary) ──
# Appends two sections to ch35_comparison.md (idempotent: earlier copies of these sections are replaced).
MARK = '\n## PAIRED comparison on identical graphs'
md_path = f'{ART}/ch35_comparison.md'
base = open(md_path).read().split(MARK)[0].rstrip('\n')
fmt3 = lambda r: f'{r.estimate:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}]' if r.kind == 'difference' else f'{r.estimate:.3f} [{r.lo:.3f}, {r.hi:.3f}]'
L = [MARK.strip('\n'), '',
     'All three models scored on EXACTLY the same test graphs: 25×25 = the 1,048 graphs outside the CNN\'s own '
     'training/validation split; 35×35 and 45×45 = full test sets. Differences use a paired bootstrap (identical '
     'resampled graphs for both models) and identical balanced subsamples. This table, not the headline table, is the '
     'basis for any GNN-vs-CNN claim. best_model and cnn_best are both zero-shot at 35/45; backbone_no55 was trained '
     'on 35/45 pool data, so its 35/45 comparison with the zero-shot CNN is not like-for-like.', '',
     '| scope | model | AUROC | balanced acc. | macro-F1 (balanced) |', '|---|---|---|---|---|']
for scope in scopes_p:
    for m in MODELS_P:
        r = {k: PAIRED[(PAIRED.kind == 'value') & (PAIRED.scope == scope) & (PAIRED.model == m) & (PAIRED.metric == k)].iloc[0] for k in NAMES}
        L.append(f'| {scope} | {m} | {fmt3(r["auroc"])} | {fmt3(r["balanced_accuracy"])} | {fmt3(r["macro_f1_balanced"])} |')
L += ['', '| scope | difference (A − B) | ΔAUROC | Δbalanced acc. | Δmacro-F1 (balanced) |', '|---|---|---|---|---|']
for scope in scopes_p:
    for pair in PAIRED[PAIRED.kind == 'difference'].model.unique():
        r = {k: PAIRED[(PAIRED.kind == 'difference') & (PAIRED.scope == scope) & (PAIRED.model == pair) & (PAIRED.metric == k)].iloc[0] for k in NAMES}
        L.append(f'| {scope} | {pair} | {fmt3(r["auroc"])} ({r["auroc"].verdict}) | {fmt3(r["balanced_accuracy"])} '
                 f'({r["balanced_accuracy"].verdict}) | {fmt3(r["macro_f1_balanced"])} ({r["macro_f1_balanced"].verdict}) |')

M = pd.read_csv(f'{ART}/ch35_metrics.csv')
sec = M[M.threshold.astype(str).str.startswith('SECONDARY') & (M.metric == 'balanced_accuracy')]
head = M[(M.threshold == 'fixed_-10dB') & (M.role == 'TEST') & (M.metric == 'balanced_accuracy')]
L += ['', '## SECONDARY: threshold calibration (VAL-fitted Youden threshold, applied unchanged to TEST)', '',
      'The pre-registered headline uses −10 dB. Youden thresholds fitted on VAL fall well above −10 dB, i.e. the models '
      'systematically predict resonance minima that are too shallow (the known depth bias). Ranking (AUROC) is unaffected; '
      'a single VAL-fitted offset recovers much of the decision accuracy. Reported for information; it never replaces the headline.', '',
      '| model | scope | VAL-fitted threshold | balanced acc. at −10 dB | balanced acc. at VAL threshold |', '|---|---|---|---|---|']
for r in sec.itertuples():
    sub = 'clean' if r.model_id == 'cnn_best' else 'all'
    h = head[(head.model_id == r.model_id) & (head.subset == sub) & (head.scope == r.scope)]
    L.append(f'| {r.model_id} | {r.scope} | {r.threshold.split("=")[1]} | '
             f'{h.estimate.iloc[0]:.3f} | {r.estimate:.3f} |' if len(h) else f'| {r.model_id} | {r.scope} | {r.threshold} | — | {r.estimate:.3f} |')
L += ['', '_Note on the CNN contamination assumption: an ~90% overlap between two independent random 90% splits is '
      'expected regardless of index mapping, so the overlap count does not validate the assumption that Chunk 3\'s '
      'dataset index equals local_idx. Weak supporting evidence: cnn_best scores higher on the in-sample 25×25 graphs '
      'than on the clean ones, the signature of genuine contamination._']
open(md_path, 'w').write(base + '\n' + '\n'.join(L) + '\n')
print('\n'.join(L))

## PAIRED comparison on identical graphs

All three models scored on EXACTLY the same test graphs: 25×25 = the 1,048 graphs outside the CNN's own training/validation split; 35×35 and 45×45 = full test sets. Differences use a paired bootstrap (identical resampled graphs for both models) and identical balanced subsamples. This table, not the headline table, is the basis for any GNN-vs-CNN claim. best_model and cnn_best are both zero-shot at 35/45; backbone_no55 was trained on 35/45 pool data, so its 35/45 comparison with the zero-shot CNN is not like-for-like.

| scope | model | AUROC | balanced acc. | macro-F1 (balanced) |
|---|---|---|---|---|
| grid_25_clean | best_model | 0.887 [0.867, 0.906] | 0.812 [0.791, 0.833] | 0.807 [0.794, 0.819] |
| grid_25_clean | backbone_no55 | 0.891 [0.872, 0.909] | 0.803 [0.782, 0.825] | 0.798 [0.787, 0.810] |
| grid_25_clean | cnn_best | 0.888 [0.868, 0.906] | 0.821 [0.801, 0.840] | 0.816 [0.804, 0.828] |
| grid_35 | best_model | 0.929 [0.904, 0.953] |

## Final guards
**What it checks:**
- exactly one test touch
- no grid 55
- headline decisions all use −10 dB
- the label definition holds on every row
- Youden thresholds are labelled SECONDARY
- the input files are byte-identical
- no earlier artifact was modified
- every output is named `ch35_*`

In [14]:
# ── Final guards ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)
guard(TEST_TOUCHES == 1, 'TEST produced exactly once (TEST TOUCH cell)')
guard(not (ALL.grid == 55).any() and not (MET_ALL.get('scope', pd.Series()).str.contains('55').any()), 'no grid 55 in any output')
guard(FIXED_TAU == -10.0 and (ALL.y_hat == (ALL.pred_min_db < -10.0)).all(), 'headline decisions use the fixed -10 dB threshold')
guard((ALL.y == (ALL.true_min_db < -10.0)).all(), 'LABEL_MIN definition holds on every row')
guard(all(t.startswith('SECONDARY') for t in pd.DataFrame(SEC).get('threshold', pd.Series(dtype=str))), 'Youden thresholds are labelled SECONDARY (VAL-fitted)')
guard(all(digest(p) == h for p, h in IMM_SHA.items()), f'input files byte-identical ({len(IMM_SHA)})')
guard(all(os.path.getmtime(p) == t for p, t in IMM_MTIME.items() if os.path.exists(p)), f'no earlier artifact modified ({len(IMM_MTIME)})')
guard(all(Path(p).name.startswith('ch35_') for p in WRITTEN), f'all {len(WRITTEN)} outputs are ch35_*')
guard(TEST_TOUCHES == 1, 'the paired/addendum cells read the parquet only (no extra test touch)')
guard(os.path.exists(f'{ART}/ch35_paired.csv'), 'paired comparison written')
assert not fails, fails
print('ALL GUARDS PASSED. No training, no optimizer, no test-based selection occurred.')

  [PASS] TEST produced exactly once (TEST TOUCH cell)
  [PASS] no grid 55 in any output
  [PASS] headline decisions use the fixed -10 dB threshold
  [PASS] LABEL_MIN definition holds on every row
  [PASS] Youden thresholds are labelled SECONDARY (VAL-fitted)
  [PASS] input files byte-identical (6)
  [PASS] no earlier artifact modified (66)
  [PASS] all 4 outputs are ch35_*
  [PASS] the paired/addendum cells read the parquet only (no extra test touch)
  [PASS] paired comparison written
ALL GUARDS PASSED. No training, no optimizer, no test-based selection occurred.
